In [1]:
import os

# Đặt giới hạn số file muốn in ra trong mỗi thư mục con
MAX_FILES_PER_DIR = 3

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(f"📁 Thư mục: {dirname}")
    
    # Chỉ lấy tối đa 3 file đầu tiên
    for filename in filenames[:MAX_FILES_PER_DIR]:
        print(f"   └── 📄 {filename}")
        
    # Nếu còn nhiều file hơn, in thêm dấu ba chấm để biết là chưa hiện hết
    if len(filenames) > MAX_FILES_PER_DIR:
        print(f"   └── 📄 ... và {len(filenames) - MAX_FILES_PER_DIR} file khác.")

In [ ]:
# 1. Cài đặt các thư viện cần thiết và gỡ cài đặt torchao bị xung đột trên Kaggle
!pip uninstall -y torchao
!pip install -q transformers datasets peft bitsandbytes accelerate jiwer soundfile librosa pandas click matplotlib tabulate

In [ ]:
# 2. Clone mã nguồn từ GitHub (nhánh benchmark/scaling-laws-results)
import os

# Để clone từ private repo, bạn có 2 cách:
# Cách 1 (Khuyên dùng): Vào Add-ons -> Secrets trên Kaggle, tạo một Secret tên "GITHUB_PAT" chứa GitHub Personal Access Token.
# Cách 2: Điền trực tiếp Token vào biến github_pat dưới đây.
github_pat = ""

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    github_pat = user_secrets.get_secret("GITHUB_PAT")
except Exception:
    pass

if github_pat:
    clone_url = f"https://{github_pat}@github.com/silvermango9927/synthetic-data-pipeline.git"
else:
    clone_url = "https://github.com/silvermango9927/synthetic-data-pipeline.git"

if not os.path.exists("synthetic-data-pipeline"):
    !git clone -b benchmark/scaling-laws-results {clone_url}
else:
    print("Codebase đã tồn tại. Tiến hành git pull để cập nhật...")
    %cd synthetic-data-pipeline
    !git remote set-url origin {clone_url}
    !git pull origin benchmark/scaling-laws-results
    %cd ..

In [ ]:
# 3. Tạo symbolic link để trỏ dataset từ Kaggle input vào đúng thư mục code mong đợi
import os
from pathlib import Path

# Thư mục gốc chứa dataset trên Kaggle
kaggle_dataset_path = "/kaggle/input/datasets/trihuynhviprovcl/synthetic-asr-zh/synthetic-asr-zh"
# Thư mục đích trong workspace code
local_target_path = "/kaggle/working/synthetic-data-pipeline/outputs/hf_datasets/synthetic-asr-zh"

# Tạo thư mục cha nếu chưa có
os.makedirs(os.path.dirname(local_target_path), exist_ok=True)

# Tạo symbolic link
if os.path.exists(local_target_path):
    if os.path.islink(local_target_path):
        os.unlink(local_target_path)
    else:
        import shutil
        shutil.rmtree(local_target_path)

os.symlink(kaggle_dataset_path, local_target_path)
print(f"🔗 Đã liên kết dataset từ Kaggle sang: {local_target_path}")

# Kiểm tra thử đường dẫn tệp tin manifest để đảm bảo kết nối hoạt động tốt
test_file = Path(local_target_path) / "data" / "long_clean" / "manifest.jsonl"
if test_file.exists():
    print("✅ Kết nối dữ liệu thành công!")
else:
    print("❌ Lỗi: Không tìm thấy tệp tin dữ liệu. Vui lòng kiểm tra lại cấu trúc dataset Kaggle input.")

In [ ]:
# 4. Chạy ASR Scaling Sweep với Whisper Large v3 Turbo trên Kaggle T4x2
%cd /kaggle/working/synthetic-data-pipeline

# Lệnh chạy quét scaling law tự động (Chạy mặc định fp16 + LoRA để tốc độ nhanh nhất)
# Nếu gặp lỗi Out-Of-Memory (OOM) VRAM, hãy thêm cờ --load-in-8bit vào bên dưới.
!python -m benchmark.scaling \
  --model-name openai/whisper-large-v3-turbo \
  --use-lora \
  --epochs 3 \
  --batch-size 4 \
  --lr 5e-5 \
  --fractions "0.1,0.2,0.4,0.6,0.8,1.0" \
  --lang zh

In [ ]:
# 5. Hiển thị biểu đồ kết quả Scaling Laws
from IPython.display import Image, display
import glob

# Tìm file biểu đồ .png được sinh ra
plot_files = glob.glob("outputs/benchmark/stats/scaling_*.png")
if plot_files:
    print(f"📈 Hiển thị biểu đồ: {plot_files[0]}")
    display(Image(filename=plot_files[0]))
else:
    print("❌ Không tìm thấy biểu đồ kết quả. Vui lòng kiểm tra xem sweep đã chạy xong chưa.")